# 🧹 Làm Sạch & Tiền Xử Lý Dữ Liệu — 20_2521001036_PJ21_LeThiPhuongAnh

**Notebook gồm 8 bước hoàn chỉnh:**
1. Import thư viện & cấu hình
2. Đọc dữ liệu
3. Thống kê lỗi **trước** khi làm sạch
4. Chuẩn hoá 5 cột phân loại
5. Trích xuất số + loại ngoại lai → NaN
6. Xoá dòng thiếu Satisfaction, Điền khuyết, Làm sạch khoảng trắng ID & văn bản
7. Xuất kết quả làm sạch, In báo cáo đối chiếu & Kiểm tra xác nhận khoảng trắng
8. **Tiền xử lý phục vụ Modeling**: One-Hot Encoding (giữ Unknown), Mã hóa nhị phân Satisfaction, Train/Test Split (80/20) & StandardScaler (chống Data Leakage)

In [1]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — Import thư viện & Cấu hình
# ══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ── Đường dẫn file (tương đối theo thư mục gốc project) ───────────
ROOT_DIR    = Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd().parents[1]
INPUT_FILE  = ROOT_DIR / "Data" / "raw" / "Du lieu khong sach.csv"
OUTPUT_FILE = ROOT_DIR / "Data" / "processed" / "PJ21.csv"
REPORT_FILE = ROOT_DIR / "outputs" / "Bao_cao_lam_sach_du_lieu.csv"

# ── Danh sách cột ─────────────────────────────────────────────────
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class", "Satisfaction"]

NUM_COLS = [
    "Age", "Flight Distance", "Departure Delay", "Arrival Delay",
    "Departure and Arrival Time Convenience", "Ease of Online Booking",
    "Check-in Service", "Online Boarding", "Gate Location",
    "On-board Service", "Seat Comfort", "Leg Room Service",
    "Cleanliness", "Food and Drink", "In-flight Service",
    "In-flight Wifi Service", "In-flight Entertainment", "Baggage Handling"
]

RATING_COLS = NUM_COLS[4:]   # 14 cột rating [0,5]

# ── Token thiếu cần nhận diện ─────────────────────────────────────
MISSING_TOKENS = {"nan", "none", "null", "n/a", "?", "-", "unknown", "chưa rõ", ""}

# ── Giới hạn hợp lệ cho các cột số ───────────────────────────────
BOUNDS = {
    "Age":             (0, 100),
    "Flight Distance": (0, 9999),
    "Departure Delay": (0, 2000),
    "Arrival Delay":   (0, 2000),
}
for rc in RATING_COLS:
    BOUNDS[rc] = (0, 5)

# ── Bảng ánh xạ chuẩn hoá phân loại ──────────────────────────────
GENDER_MAP = {"male": "Male", "female": "Female"}

CUSTOMER_TYPE_MAP = {
    "first-time": "First-time Customer",
    "first time": "First-time Customer",
    "first-time customer": "First-time Customer",
    "returning": "Returning Customer",
    "returnin": "Returning Customer",
    "returning customer": "Returning Customer",
}

TRAVEL_TYPE_MAP = {
    "business": "Business Travel",
    "busine": "Business Travel",
    "business travel": "Business Travel",
    "personal": "Personal Travel",
    "person": "Personal Travel",
    "personal travel": "Personal Travel",
}

CLASS_MAP = {
    "business": "Business",
    "economy": "Economy",
    "economy plus": "Economy Plus",
    "economy pl": "Economy Plus",
    "economy+": "Economy Plus",
}

SATISFACTION_MAP = {
    "satisfied": "Satisfied",
    "neutral or dissatisfied": "Neutral or Dissatisfied",
    "neutral or dissatisfi": "Neutral or Dissatisfied",
    "dissatisfied": "Neutral or Dissatisfied",
    "neutral": "Neutral or Dissatisfied",
}

ALL_MAPS = {
    "Gender": GENDER_MAP,
    "Customer Type": CUSTOMER_TYPE_MAP,
    "Type of Travel": TRAVEL_TYPE_MAP,
    "Class": CLASS_MAP,
    "Satisfaction": SATISFACTION_MAP,
}

# ── Regex patterns ─────────────────────────────────────────────────
UNIT_PATTERN   = re.compile(r"(?i)\b(usd|vnd|kg|%|đ)\b", re.UNICODE)
NUMBER_PATTERN = re.compile(r"[-+]?\d*\.?\d+")

# ── Hàm tiện ích ──────────────────────────────────────────────────
def is_missing_str(val) -> bool:
    """Trả về True nếu val là NaN thực hoặc chuỗi thiếu."""
    if pd.isna(val):
        return True
    return str(val).strip().lower() in MISSING_TOKENS

def has_unit_suffix(s: str) -> bool:
    """Kiểm tra chuỗi có chứa hậu tố đơn vị không."""
    return bool(UNIT_PATTERN.search(s))

def extract_number(val):
    """Trích xuất số đầu tiên trong chuỗi; trả về np.nan nếu không tìm thấy."""
    s = str(val).strip()
    if s.lower() in MISSING_TOKENS:
        return np.nan
    m = NUMBER_PATTERN.search(s)
    return float(m.group()) if m else np.nan

def normalize_cat(val, mapping: dict):
    """Chuẩn hoá giá trị phân loại theo bảng ánh xạ (hỗ trợ chuỗi bị cắt cụt)."""
    if is_missing_str(val):
        return np.nan
    cleaned = str(val).strip().lower()
    if cleaned in mapping:
        return mapping[cleaned]
    for key, std in mapping.items():
        if cleaned.startswith(key) or key.startswith(cleaned):
            return std
    return np.nan

print("✅ Cell 1 — Import & cấu hình hoàn tất.")

✅ Cell 1 — Import & cấu hình hoàn tất.


In [2]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — Đọc dữ liệu
# ══════════════════════════════════════════════════════════════════

df_raw = pd.read_csv(INPUT_FILE, dtype={"ID": str})

# Xoá cột index tự sinh (Unnamed) nếu có
df_raw = df_raw.loc[:, ~df_raw.columns.str.match(r'^Unnamed')]
df_raw.columns = df_raw.columns.str.strip()

df = df_raw.copy()

print(f"📂 File đã đọc: {INPUT_FILE}")
print(f"   Số dòng : {len(df):,}")
print(f"   Số cột  : {len(df.columns)}")
print()
print("Tên cột:")
for c in df.columns:
    print(f"  • {c}")
print()
df.head(5)

📂 File đã đọc: c:\Users\Admin\OneDrive - UFM\Documents\Lập trình python 🍀\PRJ 21\Data\raw\Du lieu khong sach.csv
   Số dòng : 129,880
   Số cột  : 24

Tên cột:
  • ID
  • Gender
  • Age
  • Customer Type
  • Type of Travel
  • Class
  • Flight Distance
  • Departure Delay
  • Arrival Delay
  • Departure and Arrival Time Convenience
  • Ease of Online Booking
  • Check-in Service
  • Online Boarding
  • Gate Location
  • On-board Service
  • Seat Comfort
  • Leg Room Service
  • Cleanliness
  • Food and Drink
  • In-flight Service
  • In-flight Wifi Service
  • In-flight Entertainment
  • Baggage Handling
  • Satisfaction



,ID,Gender,Age,Customer Type,Type of Travel,Class,Flight Distance,Departure Delay,Arrival Delay,Departure and Arrival Time Convenience,...,On-board Service,Seat Comfort,Leg Room Service,Cleanliness,Food and Drink,In-flight Service,In-flight Wifi Service,In-flight Entertainment,Baggage Handling,Satisfaction
0,1,Male,48,First-time,Business,Business,821,2,5,3,...,3,5,2,NaN,5,5 USD,3,5,5,Neutral or Dissatisfied
1,2,Female,35,unknown,Business,Business,821,NaN,39,2,...,5 đ,4,5 VND,5,3,5,2,5,5 VND,Satisfied
2,3,Male,41,Returning,NaN,Business,853,0,0,4,...,3,5 đ,3,NaN,5,NaN,4,3,3,?
3,4,Male,50,NaN,NaN,Business,1905,-,0,2,...,5,-1,5,4,4,5,2 USD,5 đ,5,Satisfied
4,5,NaN,49,?,BUSINESS,Business,NaN,0,1,3,...,3,NaN,4 đ,5,4,3 %,3,3,3 USD,Satisfied


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — Thống kê lỗi TRƯỚC khi làm sạch
# ══════════════════════════════════════════════════════════════════

def count_missing_errors(series: pd.Series) -> int:
    """Đếm ô thiếu (NaN thật + chuỗi thiếu) qua value_counts."""
    val_counts = series.value_counts(dropna=False)
    total = 0
    for val, count in val_counts.items():
        if is_missing_str(val):
            total += count
    return total

def count_format_errors_cat(series: pd.Series, mapping: dict) -> int:
    """Đếm ô phân loại có lỗi định dạng/chính tả."""
    val_counts = series.value_counts(dropna=True)
    total = 0
    for val, count in val_counts.items():
        if is_missing_str(val):
            continue
        raw = str(val)
        cleaned = raw.strip().lower()
        matched_key = None
        for key in mapping:
            if cleaned == key or cleaned.startswith(key) or key.startswith(cleaned):
                matched_key = key
                break
        if matched_key is None:
            continue
        if raw.strip() != mapping[matched_key]:
            total += count
    return total

def count_format_errors_num(series: pd.Series) -> int:
    """Đếm ô số có hậu tố đơn vị hoặc khoảng trắng thừa."""
    val_counts = series.value_counts(dropna=True)
    total = 0
    for val, count in val_counts.items():
        if is_missing_str(val):
            continue
        s = str(val).strip()
        if has_unit_suffix(s) or str(val) != s:
            total += count
    return total

def count_outliers(series: pd.Series, col_name: str) -> int:
    """Đếm giá trị ngoại lai/sai logic sau khi trích xuất số."""
    if col_name not in BOUNDS:
        return 0
    lo, hi = BOUNDS[col_name]
    val_counts = series.value_counts(dropna=True)
    total = 0
    for val, count in val_counts.items():
        num = extract_number(val)
        if pd.isna(num):
            continue
        if col_name in ("Departure Delay", "Arrival Delay"):
            if num < 0 or num > hi:
                total += count
        elif col_name == "Age":
            if num <= lo or num > hi:
                total += count
        elif col_name == "Flight Distance":
            if num <= lo or num >= hi:
                total += count
        else:  # rating [0,5]
            if num < lo or num > hi:
                total += count
    return total

# ── Tính toán ─────────────────────────────────────────────────────
before_records = []
for col in df.columns:
    if col == "ID":
        id_fmt_errors = int((df["ID"].astype(str).str.strip() != df["ID"].astype(str)).sum())
        before_records.append({"Cột": col, "Thiếu": 0, "Sai định dạng": id_fmt_errors, "Ngoại lai": 0})
        continue
    miss = count_missing_errors(df[col])
    if col in CAT_COLS:
        fmt = count_format_errors_cat(df[col], ALL_MAPS[col])
        out = 0
    else:
        fmt = count_format_errors_num(df[col])
        out = count_outliers(df[col], col)
    before_records.append({"Cột": col, "Thiếu": miss, "Sai định dạng": fmt, "Ngoại lai": out})

df_before = pd.DataFrame(before_records)
df_before["Tổng lỗi trước"] = df_before["Thiếu"] + df_before["Sai định dạng"] + df_before["Ngoại lai"]

print("=" * 70)
print("THỐNG KÊ LỖI TRƯỚC KHI LÀM SẠCH")
print("=" * 70)
display(df_before.style
    .set_properties(**{"text-align": "right"})
    .set_properties(subset=["Cột"], **{"text-align": "left"})
    .highlight_max(subset=["Tổng lỗi trước"], color="#f28b82")
    .format({"Thiếu": "{:,}", "Sai định dạng": "{:,}", "Ngoại lai": "{:,}", "Tổng lỗi trước": "{:,}"})
    .set_caption(f"Tổng cộng: {df_before['Tổng lỗi trước'].sum():,} lỗi trên {len(df):,} dòng")
)

In [3]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — Chuẩn hoá 5 cột phân loại
# ══════════════════════════════════════════════════════════════════

print("Trước chuẩn hoá — giá trị unique của từng cột phân loại:")
for col in CAT_COLS:
    unique_vals = df[col].dropna().str.strip().unique()
    print(f"  [{col}]: {sorted(set(str(v) for v in unique_vals))[:15]}")

# ── Áp dụng chuẩn hoá (dùng mapping trên unique values) ────────────
for col, mapping in ALL_MAPS.items():
    uniq_map = {u: normalize_cat(u, mapping) for u in df[col].dropna().unique()}
    df[col] = df[col].map(uniq_map)

print()
print("Sau chuẩn hoá — giá trị unique của từng cột phân loại:")
for col in CAT_COLS:
    unique_vals = df[col].dropna().unique()
    null_count  = df[col].isna().sum()
    print(f"  [{col}]: {sorted(unique_vals)}  |  NaN còn lại: {null_count:,}")

print()
print("✅ Cell 4 — Chuẩn hoá phân loại hoàn tất.")

Trước chuẩn hoá — giá trị unique của từng cột phân loại:
  [Gender]: ['?', 'FEMALE', 'Fema', 'Female', 'MALE', 'Male', 'chưa rõ', 'female', 'male', 'unknown']
  [Customer Type]: ['?', 'FIRST-TIME', 'First-ti', 'First-time', 'RETURNING', 'Returnin', 'Returning', 'chưa rõ', 'first-time', 'returning', 'unknown']
  [Type of Travel]: ['?', 'BUSINESS', 'Busine', 'Business', 'PERSONAL', 'Person', 'Personal', 'business', 'chưa rõ', 'personal', 'unknown']
  [Class]: ['?', 'BUSINESS', 'Business', 'ECONOMY', 'ECONOMY PLUS', 'Economy', 'Economy Pl', 'Economy Plus', 'business', 'chưa rõ', 'economy', 'economy plus', 'unknown']
  [Satisfaction]: ['?', 'NEUTRAL OR DISSATISFIED', 'Neutral or Dissatisfi', 'Neutral or Dissatisfied', 'SATISFIED', 'Satisfied', 'chưa rõ', 'neutral or dissatisfied', 'satisfied', 'unknown']

Sau chuẩn hoá — giá trị unique của từng cột phân loại:
  [Gender]: ['Female', 'Male']  |  NaN còn lại: 10,347
  [Customer Type]: ['First-time Customer', 'Returning Customer']  |  NaN còn 

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — Trích xuất số bằng Regex + Loại ngoại lai → NaN
# ══════════════════════════════════════════════════════════════════

outlier_summary = []

for col in NUM_COLS:
    uniq_num_map = {u: extract_number(u) for u in df[col].dropna().unique()}
    df[col] = df[col].map(uniq_num_map)

    # Loại ngoại lai → NaN
    outlier_count = 0
    if col in BOUNDS:
        lo, hi = BOUNDS[col]
        if col in ("Departure Delay", "Arrival Delay"):
            mask = (df[col] < 0) | (df[col] > hi)
        elif col == "Age":
            mask = (df[col] <= lo) | (df[col] > hi)
        elif col == "Flight Distance":
            mask = (df[col] <= lo) | (df[col] >= hi)
        else:  # rating
            mask = (df[col] < lo) | (df[col] > hi)
        outlier_count = int(mask.sum())
        df.loc[mask, col] = np.nan

    after_null = int(df[col].isna().sum())
    outlier_summary.append({"Cột": col, "Ngoại lai → NaN": outlier_count, "NaN sau bước này": after_null})

df_out_summary = pd.DataFrame(outlier_summary)
print("=" * 60)
print("KẾT QUẢ TRÍCH XUẤT SỐ & LOẠI NGOẠI LAI")
print("=" * 60)
display(df_out_summary.style
    .format({"Ngoại lai → NaN": "{:,}", "NaN sau bước này": "{:,}"})
    .highlight_max(subset=["Ngoại lai → NaN"], color="#f28b82")
    .set_properties(**{"text-align": "right"})
    .set_properties(subset=["Cột"], **{"text-align": "left"})
)

total_outliers = df_out_summary["Ngoại lai → NaN"].sum()
print(f"\nTổng ngoại lai đã loại: {total_outliers:,}")
print("✅ Cell 5 — Trích xuất số & loại ngoại lai hoàn tất.")

In [4]:
# ══════════════════════════════════════════════════════════════════
# CELL 6 — Xoá dòng thiếu Satisfaction, Điền khuyết & Làm sạch khoảng trắng
# ══════════════════════════════════════════════════════════════════

# ── Bước 6a: Xoá dòng thiếu biến mục tiêu Satisfaction ───────────
before_drop = len(df)
df = df[df["Satisfaction"].notna()].copy()
dropped = before_drop - len(df)
print(f"🗑  Xoá dòng thiếu Satisfaction: {dropped:,} dòng")
print(f"   Còn lại: {len(df):,} dòng\n")

# ── Bước 6b: Điền khuyết ──────────────────────────────────────────
impute_log = []

# Mode cho Gender, Class
for col in ["Gender", "Class"]:
    null_before = int(df[col].isna().sum())
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    impute_log.append({"Cột": col, "Phương pháp": "Mode", "Giá trị điền": mode_val, "Số ô đã điền": null_before})

# 'Unknown' cho Customer Type, Type of Travel
for col in ["Customer Type", "Type of Travel"]:
    null_before = int(df[col].isna().sum())
    df[col] = df[col].fillna("Unknown")
    impute_log.append({"Cột": col, "Phương pháp": "Hằng số", "Giá trị điền": "Unknown", "Số ô đã điền": null_before})

# Median → int64 cho 18 cột số
for col in NUM_COLS:
    null_before = int(df[col].isna().sum())
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val).round(0).astype("int64")
    impute_log.append({"Cột": col, "Phương pháp": "Median → int64", "Giá trị điền": str(int(median_val)), "Số ô đã điền": null_before})

# ── Bước 6c: Làm sạch cột ID & Toàn bộ các cột văn bản ─────────
# Xoá sạch khoảng trắng thừa ở đầu/cuối của ID rồi ép về kiểu int64
df["ID"] = df["ID"].astype(str).str.strip().astype("int64")
print("✨ Đã làm sạch khoảng trắng cột ID và ép kiểu int64.")

# Quét toàn bộ cột văn bản bằng .str.strip().str.replace(r'\s+', ' ', regex=True)
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
print(f"✨ Đã làm sạch khoảng trắng thừa và kép ở các cột văn bản: {list(text_cols)}\n")

# ── Hiển thị log điền khuyết ──────────────────────────────────────
df_impute = pd.DataFrame(impute_log)
print("=" * 65)
print("BẢNG ĐIỀN KHUYẾT")
print("=" * 65)
display(df_impute.style
    .set_properties(**{"text-align": "left"})
    .set_properties(subset=["Số ô đã điền"], **{"text-align": "right"})
    .format({"Số ô đã điền": "{:,}"})
    .highlight_between(subset=["Số ô đã điền"], left=1, color="#fff3cd")
)

remaining_nan = df.isna().sum().sum()
print(f"\n📊 Tổng số giá trị NaN còn lại: {remaining_nan}")
print("✅ Cell 6 — Xoá dòng, Điền khuyết & Làm sạch khoảng trắng hoàn tất.")

🗑  Xoá dòng thiếu Satisfaction: 10,788 dòng
   Còn lại: 119,092 dòng



TypeError: Cannot perform reduction 'median' with string dtype

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 7 — Xuất kết quả, In báo cáo đối chiếu & Kiểm tra xác nhận
# ══════════════════════════════════════════════════════════════════

# ── Thống kê lỗi SAU làm sạch ────────────────────────────────────
after_records = []
for col in df.columns:
    if col == "ID":
        after_records.append({"Cột": col, "Thiếu (sau)": 0, "Sai định dạng (sau)": 0, "Ngoại lai (sau)": 0})
        continue
    miss = int(df[col].isna().sum())
    after_records.append({"Cột": col, "Thiếu (sau)": miss, "Sai định dạng (sau)": 0, "Ngoại lai (sau)": 0})

df_after = pd.DataFrame(after_records)
df_after["Tổng lỗi sau"] = df_after["Thiếu (sau)"] + df_after["Sai định dạng (sau)"] + df_after["Ngoại lai (sau)"]

# ── Bảng đối chiếu ───────────────────────────────────────────────
df_report = pd.merge(df_before, df_after, on="Cột")
df_report = df_report[[
    "Cột",
    "Thiếu", "Sai định dạng", "Ngoại lai", "Tổng lỗi trước",
    "Thiếu (sau)", "Sai định dạng (sau)", "Ngoại lai (sau)", "Tổng lỗi sau"
]]
df_report["Lỗi đã xử lý"] = df_report["Tổng lỗi trước"] - df_report["Tổng lỗi sau"]

# ── Lưu file với encoding utf-8-sig ──────────────────────────────
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
df_report.to_csv(REPORT_FILE, index=False, encoding="utf-8-sig")
print(f"💾 Dữ liệu sạch   → {OUTPUT_FILE}")
print(f"💾 Báo cáo lỗi    → {REPORT_FILE}")
print()

# ── In bảng đối chiếu ─────────────────────────────────────────────
fmt_cols = ["Thiếu","Sai định dạng","Ngoại lai","Tổng lỗi trước",
            "Thiếu (sau)","Sai định dạng (sau)","Ngoại lai (sau)","Tổng lỗi sau","Lỗi đã xử lý"]
fmt_dict = {c: "{:,}" for c in fmt_cols}

total_before = df_report["Tổng lỗi trước"].sum()
total_after  = df_report["Tổng lỗi sau"].sum()
total_fixed  = df_report["Lỗi đã xử lý"].sum()

print("=" * 70)
print("BẢNG ĐỐI CHIẾU SỐ LƯỢNG LỖI TRƯỚC VÀ SAU KHI LÀM SẠCH")
print("=" * 70)

styled = (df_report.style
    .format(fmt_dict)
    .set_properties(**{"text-align": "right"})
    .set_properties(subset=["Cột"], **{"text-align": "left"})
    .applymap(lambda v: "background-color: #d9ead3" if isinstance(v, (int, float)) and v == 0 else "",
              subset=["Tổng lỗi sau"])
    .applymap(lambda v: "background-color: #f4cccc" if isinstance(v, (int, float)) and v > 0 else "",
              subset=["Tổng lỗi trước"])
    .set_caption(
        f"Tổng lỗi trước: {total_before:,}  |  Sau: {total_after:,}  |  Đã xử lý: {total_fixed:,}"
    )
)
display(styled)

# ── Kiểm tra xác nhận khoảng trắng trên file CSV đã lưu ──────────
print("\n" + "=" * 70)
print("KIỂM TRA XÁC NHẬN KHOẢNG TRẮNG ĐẦU/CUỐI DÒNG TRÊN FILE CSV ĐÃ LƯU")
print("=" * 70)
df_check = pd.read_csv(OUTPUT_FILE, dtype={"ID": str})
leading_id = int(df_check["ID"].str.startswith(" ").sum())
trailing_id = int(df_check["ID"].str.endswith(" ").sum())
print(f"  • Cột ID: {leading_id} dòng có khoảng trắng đầu, {trailing_id} dòng có khoảng trắng cuối.")

text_check_cols = df_check.select_dtypes(include=["object", "string"]).columns
for c in text_check_cols:
    lead_c = int(df_check[c].astype(str).str.startswith(" ").sum())
    trail_c = int(df_check[c].astype(str).str.endswith(" ").sum())
    double_space_c = int(df_check[c].astype(str).str.contains(r"\s{2,}", regex=True).sum())
    print(f"  • Cột '{c}': {lead_c} đầu dòng | {trail_c} cuối dòng | {double_space_c} khoảng trắng kép")

assert leading_id == 0, "Vẫn còn khoảng trắng ở đầu dòng cột ID!"
print("\n  ===> XÁC NHẬN HOÀN TOÀN: Số dòng còn khoảng trắng đầu dòng ở cột ID BẰNG 0!")

# ── Tổng kết ─────────────────────────────────────────────────────
print()
print("─" * 60)
print(f"  📌 Dòng ban đầu      : {len(df_raw):,}")
print(f"  🗑  Dòng đã xoá       : {len(df_raw) - len(df):,}  (thiếu Satisfaction)")
print(f"  ✅ Dòng sạch xuất    : {len(df):,}")
print(f"  🐛 Tổng lỗi trước    : {total_before:,}")
print(f"  🔧 Lỗi đã xử lý     : {total_fixed:,}")
print(f"  ✨ Lỗi còn lại       : {total_after:,}")
print("─" * 60)
print("🎉 HOÀN THÀNH LÀM SẠCH DỮ LIỆU!")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 — Tiền xử lý phục vụ Machine Learning Modeling
# ══════════════════════════════════════════════════════════════════

try:
    from sklearn.model_selection._split import train_test_split
except Exception:
    from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

OUTPUT_MODEL_FILE = r"c:\Lập trình python\Data Cleaning\20_2521001036_PJ21_LeThiPhuongAnh_MODEL_READY.csv"

# 1. Mã hóa nhị phân biến mục tiêu Satisfaction (Satisfied: 1, Neutral or Dissatisfied: 0)
target_map = {"Satisfied": 1, "Neutral or Dissatisfied": 0}
y = df["Satisfaction"].map(target_map).astype("int64")
print("🎯 Phân phối biến mục tiêu Satisfaction:")
print(f"   • Neutral or Dissatisfied (0): {(y == 0).sum():,} ({(y == 0).mean() * 100:.2f}%)")
print(f"   • Satisfied (1)              : {(y == 1).sum():,} ({(y == 1).mean() * 100:.2f}%)")

# 2. Loại bỏ ID và Satisfaction khỏi tập đặc trưng X
X = df.drop(columns=["ID", "Satisfaction"]).copy()

# 3. One-Hot Encoding cho các biến phân loại (Bảo toàn 100% mẫu chứa nhãn Unknown)
cat_features = ["Gender", "Customer Type", "Type of Travel", "Class"]
num_features = [c for c in X.columns if c not in cat_features]
X_encoded = pd.get_dummies(X, columns=cat_features, dtype=int)
print(f"\n🔤 Số lượng đặc trưng sau One-Hot Encoding: {X_encoded.shape[1]} (gồm 18 biến số + 11 cột dummy)")

# 4. Chia tập dữ liệu Train / Test (80 / 20) trước khi scaling để chống rò rỉ dữ liệu
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\n✂️ Kích thước phân chia Train / Test (80/20):")
print(f"   • X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"   • X_test : {X_test.shape}  |  y_test : {y_test.shape}")

# 5. Chuẩn hóa thang đo (StandardScaler)
# Fit CHỈ trên tập Train, sau đó transform cả Train và Test
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])
X_test_scaled[num_features]  = scaler.transform(X_test[num_features])
print("\n⚖️ Đã fit_transform StandardScaler trên X_train và transform trên X_test.")

# 6. Xuất tập dữ liệu toàn phần MODEL_READY đã chuẩn hóa
X_all_scaled = X_encoded.copy()
X_all_scaled[num_features] = scaler.transform(X_encoded[num_features])
df_model_ready = X_all_scaled.copy()
df_model_ready["Satisfaction"] = y
df_model_ready.to_csv(OUTPUT_MODEL_FILE, index=False, encoding="utf-8-sig")
print(f"\n💾 Đã lưu dữ liệu sẵn sàng huấn luyện mô hình ra file: {OUTPUT_MODEL_FILE}")
print(f"   • Kích thước: {df_model_ready.shape[0]:,} dòng × {df_model_ready.shape[1]} cột")
display(df_model_ready.head(5))
